In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Appendix to policy gradient I: the proofs

## Where this fits

- **For your own review**, strongly recommended: the main session states the policy gradient theorem
    - this appendix proves it
- **Position:** Part B, session 1 (the theorem), appendix
- **What is here**
    - **the log-derivative trick** (also called the likelihood-ratio trick): the gradient of an expectation becomes an expectation of a gradient
    - **the derivation of the theorem** (episode-return variant): the environment's transition probabilities drop out
    - **why the return-to-go variant is equal in expectation**: the score function $\nabla_\theta \log \pi_\theta$ has zero mean
    - **an optional second route** to the return-to-go variant, from per-step rewards (last section)
- **Why read it:** the zero-mean fact is used next session (baselines), and the log-derivative form is referred back to later
- **You need:** the objective $J(\theta)$ and the chained episode probability from the main session


## The likelihood ratio trick

**Why this section?** The main session's theorem needs one tool: turning the gradient of an expectation into an expectation of a gradient. It is called the log-derivative trick, or the likelihood-ratio trick.

It holds for *any* distribution; the next section applies it to episodes.

### Likelihood ratio trick

The likelihood ratio trick states that 
- for a parameterized probability distribution $p_\theta(x)$ 
- and a function $f(x)$:

the gradient of an expectation can be converted into an expectation of $f$ times the gradient of the log-probability (the *score function*).

It is a simple consequence of the Derivative of a Log rule of calculus.
- conditions: $f$ must not depend on $\theta$
- $p_\theta > 0$ where the log is taken
- the gradient may be moved inside the integral (regularity)
- for episodes the integral is a sum over $\traj$


$$
\begin{array}{llll} \\
\nabla_\theta \E_{x \sim p_\theta}[f(x)] & = & \nabla_\theta \int p_\theta(x) f(x) dx  & \text{convert expectation to integral}\\
& = &\int \nabla_\theta p_\theta(x) f(x) dx & \text{move grad inside the integral} \\
& = &\int f(x) \nabla_\theta p_\theta(x)  dx & \text{rearrange term} \\
& = &\int f(x) \left(p_\theta(x) \nabla_\theta \log p_\theta(x) \right)  dx & \text{log trick: } \\
& & & \nabla_\theta p_\theta(x) = p_\theta(x) \nabla_\theta \log p_\theta(x) \\
& = & \E_{x \sim p_\theta} \left[ f(x) \nabla_\theta \log p_\theta(x) \right] & \text{convert integral back to expectation} \\
\end{array}
$$

The "log trick" follows from the rules of calculus
$$
\begin{array}{llll} \\
\nabla_\theta \log p_\theta(x) & = & \frac{1}{p_\theta(x)} * \nabla_\theta p_\theta(x) & \text{Calculus: grad of log, chain rule} \\
\nabla_\theta  p_\theta(x) & = &  p_\theta(x) * \nabla_\theta  \log p_\theta(x) & \text{re-arranging terms} \\
\end{array}
$$

## Computing $\nabla_\theta J(\theta)$: the trick applied to an episode

**Why this section?** The trick of the last section holds for any distribution. Applying it to the distribution of *episodes* brings in one new fact, and that fact is what the theorem needs.

> **The new fact: the probability of an episode is a product of per-step factors. Taking the log turns that product into a sum, and the environment's unknown factors then drop out of the gradient.**

- **Step A: apply the trick** (no new proof; the mapping below is all that changes)
- **Step B: write $\log \prc{\traj}{\theta}$ using the product** (the new idea)
- **Step C: assemble** the theorem

<table>
<thead><tr><th nowrap><div align="left">Last section</div></th><th nowrap><div align="left">This section</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">the random thing $x$</div></td><td nowrap><div align="left">an episode $\traj$</div></td></tr>
<tr><td nowrap><div align="left">its distribution $p_\theta(x)$</div></td><td nowrap><div align="left">the probability of the episode, $\prc{\traj}{\theta}$</div></td></tr>
<tr><td nowrap><div align="left">the function $f(x)$</div></td><td nowrap><div align="left">the episode return $\rlret_{0,\traj}$</div></td></tr>
<tr><td nowrap><div align="left">the integral over $x$</div></td><td nowrap><div align="left">the sum over episodes $\traj$</div></td></tr>
</tbody></table>


**Step A: apply the trick** (the same steps as before, with the mapping of the previous slide)

The expectation of the episode return is a sum over episodes weighted by their probabilities. Take its gradient:

$$
\begin{array}{llll} \\
\nabla_\theta
\sum_{\traj \sim \prc{\traj}{\theta}} \prc{\traj}{\theta} *  { \rlret_{0,\traj} } 
& = & \sum_{\traj \sim \prc{\traj}{\theta}}{ \nabla_\theta  \, \,\prc{\traj}{\theta} *  { \rlret_{0,\traj} }}  & \text{grad of a sum is sum of the grads} \\ 
& = & \sum_{\traj \sim \prc{\traj}{\theta}}{ (\prc{\traj}{\theta} \nabla_\theta  \log  \prc{\traj}{\theta}) \, \,\rlret_{0,\traj} } & \text{Likelihood ratio trick:} \\
& & & \nabla_\theta \prc{\traj}{\theta} = \prc{\traj}{\theta} * \nabla_\theta \log \prc{\traj}{\theta} \\
\end{array}
$$

This is $\nabla_\theta J(\theta) = \Exp{\traj \sim \prc{\traj}{\theta}}\left[ \nabla_\theta \log \prc{\traj}{\theta} \; \rlret_{0,\traj} \right]$.

- one piece is still unknown: $\nabla_\theta \log \prc{\traj}{\theta}$
- the next slide computes it


**Step B: the product becomes a sum** (the new idea)

- the probability of an episode is a *product*, one factor per step (derived in the main session)
- differentiating a product directly needs the product rule
    - one term per factor, each multiplied by all the other factors
- the log turns the product into a *sum*, and the gradient of a sum is the sum of the gradients
    - a factor that does not depend on $\theta$ has gradient $0$ and drops out

Substitute the chained probability into $\log \prc{\traj}{\theta}$:



$$
\begin{array}{llll} \\
\log \prc{\traj}{\theta} & = & \log \left( 
\pr{\stateseq_{\traj,0}}
*
\prod_{\tt=0}^{|\traj|-1} { \pi(\actseq_{\traj,\tt} | \stateseq_{\traj,\tt}) * \transp(\stateseq_{\traj, \tt+1}, \rewseq_{\traj,\tt+1} | \stateseq_{\traj,\tt}, \actseq_{\traj,\tt})}  \right) & \text{definition of } \prc{\traj}{\theta} \\
& = & 
\log \pr{\stateseq_{\traj,0}}
+
  \sum_{\tt=0}^{|\traj|-1} {
  \left( \,
\log \pi(\actseq_{\traj,\tt} | \stateseq_{\traj,\tt}) + \log \transp(\stateseq_{\traj, \tt+1}, \rewseq_{\traj,\tt+1} | \stateseq_{\traj,\tt}, \actseq_{\traj,\tt})
\, \right)
} & \text{log of product is sum of logs}  \\
\\
\nabla_\theta \log \prc{\traj}{\theta} & = & 
\sum_{\tt=0}^{|\traj|-1} {
\nabla_\theta  \log \pi(\actseq_{\traj,\tt} | \stateseq_{\traj,\tt}) 
}  & \text{grad of sum is sum of grads} \\
& & &  \text{Transition Probability terms } \\
& & & \;\;\;\transp( \ldots) \\
& & & \text{are independent of } \theta \\
\end{array}
$$

The initial-state probability $\pr{\stateseq_{\traj,0}}$ does not depend on $\theta$ either, so it also drops out.


**Step C: assemble.** The whole proof in one line each:

$$
\begin{array}{llll} \\
\nabla_\theta \Exp{\traj \sim \pi_\theta} \,\, { \rlret_{0,\traj} } 
& = & \nabla_\theta \sum_\traj \prc{\traj}{\theta} *  { \rlret_{0,\traj} } & \text{turn the expectation into a sum weighted by episode probabilities} \\
& = & \sum_\traj{ \left(\prc{\traj}{\theta} \nabla_\theta  \log  \prc{\traj}{\theta} \right) \, \, *\rlret_{0,\traj} } & \text{Likelihood ratio trick} \\
& = & \sum_\traj{ }{ \left( \prc{\traj}{\theta} * 
    \sum_{\tt=0}^{|\traj|-1} {
\nabla_\theta  \log \pi(\actseq_{\traj,\tt} | \stateseq_{\traj,\tt}) } \right)
*  { \rlret_{0,\traj} }
} & \text{Transition Probability terms disappear from within the grad} \\
& & & \text{as shown above}\\
& = & \Exp{\traj \sim \prc{\traj}{\theta}}
\left[
\sum_{\tt=0}^{|\traj|-1} {
\nabla_\theta  \log \pi(\actseq_{\traj,\tt} | \stateseq_{\traj,\tt})  }
\; \rlret_{0,\traj} \right]
& \text{turn back into an expectation} \\
& & &\text{involving each step of the episode} \\
\end{array}
$$

## Why the return-to-go gives the same expected gradient

**Why this section?** The purpose: to prove that the two forms of the policy gradient theorem are equal on average.

> **Claim: weighting each gradient term by the return-to-go $\disc^\tt \rlret_{\tt,\traj}$ instead of the whole-episode return $\rlret_{0,\traj}$ does not change the expected gradient.**

$$
\Exp{\traj \sim \prc{\traj}{\theta}} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \cdot \rlret_{0,\traj} \right]
= \Exp{\traj \sim \prc{\traj}{\theta}} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \cdot \disc^\tt \rlret_{\tt,\traj} \right]
$$

- **whole-episode form (left):** every gradient term is weighted by one number, the return of the whole episode
- **per-step form (right):** each term is weighted only by the rewards from that step on
- the two sums are *different* random quantities for the same episode
- they are equal only *after averaging*
- the return-to-go is the discounted reward from step $\tt$ onward:
  $\rlret_{\tt,\traj} = \sum_{k=\tt}^{|\traj|-1} \disc^{k-\tt} \rewseq_{k+1}$
- **why it matters:** the whole-episode return includes rewards received *before* the action
    - they add noise and no signal
    - dropping them lowers the variance (next session)
- the plan: one lemma, a split of the return, then take expectations

**On the surface the two forms look different**

> **The two forms differ only by terms where a reward comes before the action. The zero-mean lemma turns those terms into no effect on average.**

- expand both sums into pairs: the gradient term at step $\tt$ times the reward at step $k$
- **whole-episode form:** every pair appears, so each reward multiplies *all* the gradient terms
- **per-step form:** only the pairs with the reward at or after the action ($k \geq \tt$)
    - a later reward appears in the sums of many earlier steps, which can look like double counting
    - but each kept pair has exactly the same weight, $\disc^k \rewseq_{k+1}$, as in the whole-episode form
- **the difference** is the pairs where the reward comes *before* the action ($k < \tt$)
    - they are in the whole-episode form and not in the per-step form
    - given the past, the score of the action has mean zero, so a reward from the past times that score averages to zero
    - the lemma below shows this

**The zero-mean score lemma: the score function has zero mean**

> **Given everything that happened up to step $\tt$, the gradient $\nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt)$ averages to zero over the action.**

$$
\E_{\actseq \sim \pi_\theta}\bigl[ \nabla_\theta \log \pi_\theta(\actseq \mid \state) \bigr]
= \sum_{\act} \pi_\theta(\act \mid \state) \, \nabla_\theta \log \pi_\theta(\act \mid \state)
= \sum_{\act} \nabla_\theta \pi_\theta(\act \mid \state)
= \nabla_\theta \sum_{\act} \pi_\theta(\act \mid \state)
= \nabla_\theta 1 = 0
$$

- the second equality is the log-derivative trick, and the last uses that probabilities sum to $1$
- for continuous actions the sum over $\act$ is an integral, and the argument is the same
- we use this lemma again in B2 (baseline), B3 (a learned baseline) and C2 (group baseline)

**Consequence: a quantity fixed before the action adds nothing in expectation**

- let $W$ be any quantity already determined when $\actseq_\tt$ is chosen
    - given that history, $\actseq_\tt$ is drawn from $\pi_\theta(\cdot \mid \stateseq_\tt)$
    - so average over $\actseq_\tt$ first, holding $W$ fixed

$$
\Exp{\traj \sim \prc{\traj}{\theta}}\bigl[ W \cdot \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \bigr]
= \Exp{\traj \sim \prc{\traj}{\theta}}\Bigl[ W \cdot \E_{\actseq_\tt \sim \pi_\theta}\bigl[ \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \bigr] \Bigr]
= \Exp{\traj \sim \prc{\traj}{\theta}}[ W \cdot 0 ] = 0
$$

- **rewards received before step $\tt$ are such a $W$**
- so is a *baseline* $b(\stateseq_\tt)$: it depends on the state, which is known before the action, but not on the action
    - subtracting it from the weight therefore leaves the expected gradient unchanged
    - next session uses this to reduce variance

**Split the episode return at step $\tt$**

- for one fixed episode, write $\rlret_\tt$ for $\rlret_{\tt,\traj}$
- the whole-episode return is the rewards before step $\tt$ plus the rewards from step $\tt$ onward
- the later rewards, shifted by $j = k - \tt$, are $\disc^\tt$ times the return-to-go

$$
\rlret_{0,\traj} = \sum_{k=0}^{|\traj|-1} \disc^k \rewseq_{k+1}
= \underbrace{\sum_{k=0}^{\tt-1} \disc^k \rewseq_{k+1}}_{P_\tt \text{ (rewards before step } \tt)}
+ \underbrace{\disc^\tt \sum_{j=0}^{|\traj|-1-\tt} \disc^j \rewseq_{\tt+j+1}}_{\disc^\tt \rlret_\tt}
$$

- for $\tt = 0$ there are no earlier rewards: $P_0 = 0$

**Take expectations**

Substitute $\rlret_{0,\traj} = P_\tt + \disc^\tt \rlret_\tt$ into the first form:

$$
\begin{aligned}
& \Exp{\traj \sim \prc{\traj}{\theta}} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \cdot \rlret_{0,\traj} \right] \\
& \quad = \sum_{\tt=0}^{|\traj|-1} \underbrace{\Exp{\traj \sim \prc{\traj}{\theta}} \bigl[ P_\tt \, \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \bigr]}_{= 0 \text{ by the consequence slide}} \\
& \quad \quad + \Exp{\traj \sim \prc{\traj}{\theta}} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \cdot \disc^\tt \rlret_\tt \right]
\end{aligned}
$$

- $P_\tt$ is fixed before $\actseq_\tt$ is chosen, so every term of the first sum is $0$
- what remains is the second form: the claim

**Putting it together**

$$
\nabla_\theta J(\theta)
= \Exp{\traj \sim \prc{\traj}{\theta}} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \cdot \rlret_{0,\traj} \right]
= \Exp{\traj \sim \prc{\traj}{\theta}} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \cdot \disc^\tt \rlret_\tt \right]
$$

- the first equality is the previous section
- the second is the claim
- **the return-to-go variant has the same mean and less noise**
- many references drop $\disc^\tt$
- it is $1$ when $\disc = 1$, and the exact gradient of the discounted objective keeps it


## Optional: a second route to the same theorem

> **Optional. Nothing later depends on this section. It reaches the return-to-go variant directly from the per-step rewards, without the detour through the episode return.**

- **You have already seen the result:** the previous section showed that the return-to-go variant equals the episode-return variant
- **This route differs only in where it starts**
    - it starts from the sum of per-step rewards $\sum_\tt \disc^\tt \rewseq_{\tt+1}$
    - causality then removes the past rewards in one step
- **It uses the same two tools:** the log-derivative trick and the zero-mean score function
- read it only if the earlier route was hard to follow
    - Steps 1-4 repeat what you have seen
    - only Step 5 is new

**Steps 1-4: as in the earlier derivation**

$
J(\theta) = \E_{\traj \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\traj|-1} \disc^{\tt} \rewseq_{\tt+1} \right]
$

- apply the log-derivative trick, then write $\log \prc{\traj}{\theta}$ as a sum (the transition terms drop out)
- the result is a double sum over steps $\tt'$ (the action) and $\tt$ (the reward):

$
\nabla_\theta J(\theta) = \E_{\traj \sim \pi_\theta} \left[ \sum_{\tt'=0}^{|\traj|-1} \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tt'} | \stateseq_{\tt'}) \; \disc^\tt \rewseq_{\tt+1} \right]
$

**Step 5: causality removes the past rewards (the new step)**

An action cannot affect earlier rewards: for $\tt < \tt'$ the reward $\rewseq_{\tt+1}$ is fixed before $\actseq_{\tt'}$ is chosen.

By the zero-mean score function fact (an earlier slide), the expected product is zero. This route therefore uses the same fact as the first.

Only the terms with $\tt' \leq \tt$ remain:

$
\nabla_\theta J(\theta) = \E_{\traj \sim \pi_\theta} \left[ \sum_{\tt'=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tt'} | \stateseq_{\tt'}) \left( \sum_{\tt=\tt'}^{|\traj|-1} \disc^\tt \rewseq_{\tt+1} \right) \right]
$

**Step 6: recognize the return-to-go**

$
\sum_{\tt=\tt'}^{|\traj|-1} \disc^\tt \rewseq_{\tt+1} = \disc^{\tt'} \sum_{j=0}^{|\traj|-1-\tt'} \disc^j \rewseq_{\tt'+j+1} = \disc^{\tt'} \rlret_{\tt'}
$

So we may write:

$
\nabla_\theta J(\theta) = \E_{\traj \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \cdot \disc^\tt \rlret_\tt \right]
$

- this is the policy gradient theorem, return-to-go variant
- many references and implementations drop $\disc^\tt$ (it is $1$ when $\disc = 1$)
- the exact gradient of the discounted objective keeps it


# What we established, and what it sets up

- **Established**
    - $\nabla_\theta p_\theta(x) = p_\theta(x) \nabla_\theta \log p_\theta(x)$: the log-derivative trick
    - the log of an episode's probability is a sum of logs
        - the transition probabilities do not depend on $\theta$
        - so $\nabla_\theta \log \prc{\traj}{\theta}$ is a sum of $\nabla_\theta \log \pi_\theta$ terms
    - hence the episode-return variant of the theorem, with no $\transp$
    - the score function has zero mean, so rewards already received do not change the expected gradient: the return-to-go variant
- **Sets up**
    - the next session subtracts a *baseline* from the weight: it keeps the expected gradient because the score function has zero mean
    - later sessions refer back to the log-derivative form of the gradient
